# ARXML Parser for Enums
This notebook parses ARXML files, extracts data type mappings, and generates a DataFrame with enumerated states.

In [11]:
# Import required libraries
import pandas as pd
from lxml import etree
import re
import os
import math
from typing import Any, Dict

In [12]:
# Define the parser function
def parse_arxml_with_enums(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (This might take a moment to build lookup tables)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    # 1. Extract all CompuMethods (The actual Enum dictionaries)
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        enums = {}
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            lower_limit = scale.xpath("*[local-name()='LOWER-LIMIT']")
            vt = scale.xpath(".//*[local-name()='VT']")
            if lower_limit and vt and lower_limit[0].text and vt[0].text:
                enums[lower_limit[0].text.strip()] = vt[0].text.strip()
        
        if enums:
            compu_methods[cm_name] = enums

    # 2. Extract Application Data Types to map them to CompuMethods
    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            ref_name = compu_ref[0].text.split("/")[-1].strip()
            app_to_compu[app_name] = ref_name

    # Helper function to format the dictionary into a readable string
    def get_enum_string(app_type_name):
        compu_name = app_to_compu.get(app_type_name)
        enum_dict = compu_methods.get(compu_name, {})
        if not enum_dict:
            return "No Enums"
        return " | ".join([f"{k}: {v}" for k, v in enum_dict.items()])

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem:
            continue
        
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match:
            continue
        
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        valid_methods = []
        valuestate_app_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    continue 
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw))
        
        for clean_method, raw_app_name in valid_methods:
            base_enums = get_enum_string(raw_app_name)
            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "Available_States": base_enums
            })
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                vs_enums = get_enum_string(valuestate_app_name)
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "Available_States": vs_enums
                })

    df = pd.DataFrame(parsed_data)
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
    
    return df

In [13]:
def parse_arxml_with_boundaries(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Building Enum and Physical Range tables)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        
        # Calculate Mid and round down to the lower base using math.floor
        mid_val = None
        if has_limits:
            mid_val = math.floor((min_val + max_val) / 2)
            
        compu_methods[cm_name] = {
            "enums": enums,
            "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None,
            "max": max_val if has_limits else None,
            "mid": mid_val
        }

    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            ref_name = compu_ref[0].text.split("/")[-1].strip()
            app_to_compu[app_name] = ref_name

    def get_compu_data(app_type_name):
        compu_name = app_to_compu.get(app_type_name)
        return compu_methods.get(compu_name, {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None
        })

    def format_val(v):
        if v is None: return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem:
            continue
            
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match:
            continue
            
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        valid_methods = []
        valuestate_app_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    continue 
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw))
        
        for clean_method, raw_app_name in valid_methods:
            c_data = get_compu_data(raw_app_name)
            
            if c_data["has_enums"]:
                states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()])
            elif c_data["min"] is not None:
                states_str = "Physical Value"
            else:
                states_str = "No Data"

            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "Available_States": states_str,
                "Min": format_val(c_data["min"]),
                "Mid": format_val(c_data["mid"]),
                "Max": format_val(c_data["max"])
            })
            
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                
                vs_c_data = get_compu_data(valuestate_app_name)
                vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "Available_States": vs_states,
                    "Min": format_val(vs_c_data["min"]),
                    "Mid": format_val(vs_c_data["mid"]),
                    "Max": format_val(vs_c_data["max"])
                })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
        
        # --- Analytics Summary ---
        physical_count = (df['Available_States'] == "Physical Value").sum()
        no_data_count = (df['Available_States'] == "No Data").sum()
        enum_count = len(df) - physical_count - no_data_count
        
        print("\n--- Parsing Summary ---")
        print(f"Total Signals Extracted : {len(df)}")
        print(f"Signals with Enums      : {enum_count}")
        print(f"Physical Value Signals  : {physical_count}")
        print(f"Signals with No Data    : {no_data_count}\n")
    
    return df

In [14]:
from IPython.display import display

# Example usage in notebook
file_name = "ETH_CAN.arxml"  # Replace with your ARXML file path
df_signals = parse_arxml_with_boundaries(file_name)

if not df_signals.empty:
    print(f"Success! Extracted {len(df_signals)} signals with Enums and Physical Boundaries.")
    
    # Configure pandas to not truncate our dataframe view
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 60)
    
    # Define the columns we want to inspect
    cols_to_view = ['Event', 'Method', 'Available_States', 'Min', 'Mid', 'Max']
    
    display(df_signals[cols_to_view].head(25))
    
    # Optionally export to CSV
    # df_signals.to_csv("signals_with_boundaries.csv", index=False)
    # print("\nData exported to signals_with_boundaries.csv")
else:
    print("No signals found or failed to parse the file.")

Parsing ETH_CAN.arxml (Building Enum and Physical Range tables)...

--- Parsing Summary ---
Total Signals Extracted : 7865
Signals with Enums      : 4620
Physical Value Signals  : 1371
Signals with No Data    : 1874

Success! Extracted 7865 signals with Enums and Physical Boundaries.


,Event,Method,Available_States,Min,Mid,Max
0,SomeIpGadeSignal,GadeStatus,0: GADESTATUS_PARK_MODE | 1: GADESTATUS_LIFE_ON_BOARD | ...,0,3,7
1,SomeIpGadeSignal,GadeStatusValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2
2,SomeIpGadeSignal,gadeEvent,No Data,N/A,N/A,N/A
3,SomeIpGadeSignal,gadeEventValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2
4,SomeIpMinVoltageReq,VoltageValueType,Physical Value,10,13,16
5,SomeIpMinVoltageReq,VoltageValueTypeValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2
6,SomeIpMinVoltageReq,minVoltageValue,Physical Value,10,13,16
7,SomeIpMinVoltageReq,minVoltageValue1,Physical Value,10,13,16
8,SomeIpMinVoltageReq,minVoltageValue1ValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2
9,SomeIpMinVoltageReq,minVoltageValue2,Physical Value,10,13,16


In [15]:
import os
import re
import math
import pandas as pd
from lxml import etree
from IPython.display import display

def parse_arxml_with_scaling(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Extracting Enums, Ranges, Scaling, and Units)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        # 1. Extract Unit
        unit_ref = cm.xpath("*[local-name()='UNIT-REF']")
        unit = unit_ref[0].text.split("/")[-1].strip() if unit_ref and unit_ref[0].text else "N/A"
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        factor = "N/A"
        offset = "N/A"
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            coeffs_node = scale.xpath(".//*[local-name()='COMPU-RATIONAL-COEFFS']")
            
            # Min / Max Boundaries
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            # Enums
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
                
            # 2. Extract Scaling Coefficients (Factor & Offset)
            if coeffs_node:
                try:
                    num_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-NUMERATOR']/*[local-name()='V']")
                    den_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-DENOMINATOR']/*[local-name()='V']")
                    
                    # AUTOSAR specifies Numerator[0] is offset, Numerator[1] is factor
                    n0 = float(num_v[0].text) if len(num_v) > 0 else 0.0
                    n1 = float(num_v[1].text) if len(num_v) > 1 else 1.0
                    d = float(den_v[0].text) if len(den_v) > 0 else 1.0
                    
                    if d != 0:
                        offset = n0 / d
                        factor = n1 / d
                except Exception:
                    pass
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        
        mid_val = None
        if has_limits:
            mid_val = math.floor((min_val + max_val) / 2)
            
        compu_methods[cm_name] = {
            "enums": enums,
            "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None,
            "max": max_val if has_limits else None,
            "mid": mid_val,
            "unit": unit,
            "factor": factor,
            "offset": offset
        }

    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            ref_name = compu_ref[0].text.split("/")[-1].strip()
            app_to_compu[app_name] = ref_name

    def get_compu_data(app_type_name):
        compu_name = app_to_compu.get(app_type_name)
        return compu_methods.get(compu_name, {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None,
            "unit": "N/A", "factor": "N/A", "offset": "N/A"
        })

    def format_val(v):
        if v is None or v == "N/A": return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem:
            continue
            
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match:
            continue
            
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        valid_methods = []
        valuestate_app_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    continue 
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw))
        
        for clean_method, raw_app_name in valid_methods:
            c_data = get_compu_data(raw_app_name)
            
            if c_data["has_enums"]:
                states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()])
            elif c_data["min"] is not None:
                states_str = "Physical Value"
            else:
                states_str = "No Data"

            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "Available_States": states_str,
                "Min": format_val(c_data["min"]),
                "Mid": format_val(c_data["mid"]),
                "Max": format_val(c_data["max"]),
                "Factor": format_val(c_data["factor"]),
                "Offset": format_val(c_data["offset"]),
                "Unit": c_data["unit"]
            })
            
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                
                vs_c_data = get_compu_data(valuestate_app_name)
                vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "Available_States": vs_states,
                    "Min": format_val(vs_c_data["min"]),
                    "Mid": format_val(vs_c_data["mid"]),
                    "Max": format_val(vs_c_data["max"]),
                    "Factor": format_val(vs_c_data["factor"]),
                    "Offset": format_val(vs_c_data["offset"]),
                    "Unit": vs_c_data["unit"]
                })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
        
        physical_count = (df['Available_States'] == "Physical Value").sum()
        no_data_count = (df['Available_States'] == "No Data").sum()
        enum_count = len(df) - physical_count - no_data_count
        
        print("\n--- Parsing Summary ---")
        print(f"Total Signals Extracted : {len(df)}")
        print(f"Signals with Enums      : {enum_count}")
        print(f"Physical Value Signals  : {physical_count}")
        print(f"Signals with No Data    : {no_data_count}\n")
    
    return df

# ==========================================
# EXECUTION BLOCK
# ==========================================
file_name = "ETH_CAN.arxml" 
df_signals = parse_arxml_with_scaling(file_name)

if not df_signals.empty:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 50)
    
    # Adding Factor, Offset, and Unit to our view
    cols_to_view = ['Method', 'Available_States', 'Min', 'Mid', 'Max', 'Factor', 'Offset', 'Unit']
    display(df_signals[cols_to_view].head(25))

Parsing ETH_CAN.arxml (Extracting Enums, Ranges, Scaling, and Units)...

--- Parsing Summary ---
Total Signals Extracted : 7865
Signals with Enums      : 4620
Physical Value Signals  : 1371
Signals with No Data    : 1874



,Method,Available_States,Min,Mid,Max,Factor,Offset,Unit
0,GadeStatus,0: GADESTATUS_PARK_MODE | 1: GADESTATUS_LIFE_O...,0,3,7,N/A,N/A,N/A
1,GadeStatusValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VA...,0,1,2,N/A,N/A,N/A
2,gadeEvent,No Data,N/A,N/A,N/A,N/A,N/A,N/A
3,gadeEventValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VA...,0,1,2,N/A,N/A,N/A
4,VoltageValueType,Physical Value,10,13,16,1,0,Volt
5,VoltageValueTypeValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VA...,0,1,2,N/A,N/A,N/A
6,minVoltageValue,Physical Value,10,13,16,1,0,Volt
7,minVoltageValue1,Physical Value,10,13,16,1,0,Volt
8,minVoltageValue1ValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VA...,0,1,2,N/A,N/A,N/A
9,minVoltageValue2,Physical Value,10,13,16,1,0,Volt


In [ ]:
import os
import re
import math
import pandas as pd
from lxml import etree
from IPython.display import display

def parse_arxml_complete(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Cleaning up Data Types)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    # 1. CompuMethods (Enums, Min/Max/Mid, Units, Scaling)
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        unit_ref = cm.xpath("*[local-name()='UNIT-REF']")
        unit = unit_ref[0].text.split("/")[-1].strip() if unit_ref and unit_ref[0].text else "N/A"
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        factor = "N/A"
        offset = "N/A"
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            coeffs_node = scale.xpath(".//*[local-name()='COMPU-RATIONAL-COEFFS']")
            
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
                
            if coeffs_node:
                try:
                    num_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-NUMERATOR']/*[local-name()='V']")
                    den_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-DENOMINATOR']/*[local-name()='V']")
                    
                    n0 = float(num_v[0].text) if len(num_v) > 0 else 0.0
                    n1 = float(num_v[1].text) if len(num_v) > 1 else 1.0
                    d = float(den_v[0].text) if len(den_v) > 0 else 1.0
                    
                    if d != 0:
                        offset = n0 / d
                        factor = n1 / d
                except Exception: pass
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        mid_val = math.floor((min_val + max_val) / 2) if has_limits else None
            
        compu_methods[cm_name] = {
            "enums": enums,
            "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None,
            "max": max_val if has_limits else None,
            "mid": mid_val,
            "unit": unit,
            "factor": factor,
            "offset": offset
        }

    # 2. Application -> CompuMethod Mapping
    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            app_to_compu[app_name] = compu_ref[0].text.split("/")[-1].strip()

    # 3. Implementation -> BaseType Mapping (WITH FIX)
    impl_to_basetype = {}
    for impl_dt in root.xpath("//*[local-name()='IMPLEMENTATION-DATA-TYPE']"):
        impl_name_elem = impl_dt.xpath("*[local-name()='SHORT-NAME']")
        base_ref = impl_dt.xpath(".//*[local-name()='BASE-TYPE-REF']")
        if impl_name_elem and base_ref and base_ref[0].text:
            impl_name = impl_name_elem[0].text.strip()
            basetype_raw = base_ref[0].text.split("/")[-1].strip()
            
            # --- THE FIX ---
            # Clean up AUTOSAR authoring tool quirks (e.g., 'uint81' -> 'uint8', 'float32_0' -> 'float32')
            clean_match = re.match(r'^(u?s?int(?:8|16|32|64)|float(?:32|64)|boolean|double)', basetype_raw, re.IGNORECASE)
            basetype_name = clean_match.group(1).lower() if clean_match else basetype_raw
            
            impl_to_basetype[impl_name] = basetype_name

    def get_compu_data(app_type_name):
        return compu_methods.get(app_to_compu.get(app_type_name), {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None,
            "unit": "N/A", "factor": "N/A", "offset": "N/A"
        })

    def format_val(v):
        if v is None or v == "N/A": return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem: continue
            
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match: continue
            
        sif = match.group(1)
        someip_event = f"SomeIp{match.group(2)}"
        
        valid_methods = []
        valuestate_app_name = None
        valuestate_impl_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            impl_ref = dt_map.xpath("*[local-name()='IMPLEMENTATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                impl_path_raw = impl_ref[0].text.split("/")[-1].strip() if impl_ref and impl_ref[0].text else None
                
                # Check for ValueState placeholders
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    valuestate_impl_name = impl_path_raw # Save implementation name for data type lookup
                    continue 
                    
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw, impl_path_raw))
        
        for clean_method, raw_app_name, raw_impl_name in valid_methods:
            c_data = get_compu_data(raw_app_name)
            datatype = impl_to_basetype.get(raw_impl_name, "N/A") if raw_impl_name else "N/A"
            
            if c_data["has_enums"]:
                states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()])
            elif c_data["min"] is not None:
                states_str = "Physical Value"
            else:
                states_str = "No Data"

            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "DataType": datatype,
                "Available_States": states_str,
                "Min": format_val(c_data["min"]),
                "Mid": format_val(c_data["mid"]),
                "Max": format_val(c_data["max"]),
                "Factor": format_val(c_data["factor"]),
                "Offset": format_val(c_data["offset"]),
                "Unit": c_data["unit"]
            })
            
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                
                vs_c_data = get_compu_data(valuestate_app_name)
                vs_datatype = impl_to_basetype.get(valuestate_impl_name, "N/A") if valuestate_impl_name else "N/A"
                vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "DataType": vs_datatype,
                    "Available_States": vs_states,
                    "Min": format_val(vs_c_data["min"]),
                    "Mid": format_val(vs_c_data["mid"]),
                    "Max": format_val(vs_c_data["max"]),
                    "Factor": format_val(vs_c_data["factor"]),
                    "Offset": format_val(vs_c_data["offset"]),
                    "Unit": vs_c_data["unit"]
                })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
    
    return df

# ==========================================
# EXECUTION BLOCK
# ==========================================
file_name = "ETH_CAN.arxml" 
df_signals = parse_arxml_complete(file_name)

if not df_signals.empty:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 40)
    
    cols_to_view = ['Method', 'DataType', 'Available_States', 'Min', 'Mid', 'Max', 'Unit']
    display(df_signals[cols_to_view].head(25))

Parsing ETH_CAN.arxml (Cleaning up Data Types)...


,Method,DataType,Available_States,Min,Mid,Max,Unit
0,GadeStatus,uint8,0: GADESTATUS_PARK_MODE | 1: GADESTA...,0,3,7,N/A
1,GadeStatusValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,1,2,N/A
2,gadeEvent,uint8,No Data,N/A,N/A,N/A,N/A
3,gadeEventValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,1,2,N/A
4,VoltageValueType,float32,Physical Value,10,13,16,Volt
5,VoltageValueTypeValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,1,2,N/A
6,minVoltageValue,float32,Physical Value,10,13,16,Volt
7,minVoltageValue1,float32,Physical Value,10,13,16,Volt
8,minVoltageValue1ValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,1,2,N/A
9,minVoltageValue2,float32,Physical Value,10,13,16,Volt


In [18]:
# 1. Overall Signal Categorization
physical_count = (df_signals['Available_States'] == "Physical Value").sum()
no_data_count = (df_signals['Available_States'] == "No Data").sum()
enum_count = len(df_signals) - physical_count - no_data_count

print("=== PARSING SUMMARY ===")
print(f"Total Signals       : {len(df_signals)}")
print(f"Signals with Enums  : {enum_count}")
print(f"Physical Signals    : {physical_count}")
print(f"Missing / No Data   : {no_data_count}\n")

# 2. Data Type Distribution (This will prove if 'double' was caught!)
print("=== DATA TYPE DISTRIBUTION ===")
print(df_signals['DataType'].value_counts())
print("\n")

# 3. Quick check on how many physical signals have Units defined
print("=== UNIT DISTRIBUTION ===")
# Filter out "N/A" to see the actual units being used
print(df_signals[df_signals['Unit'] != "N/A"]['Unit'].value_counts())
print("\n")

# 4. View a sample of the "No Data" signals to investigate them
print("=== NO DATA SAMPLE ===")
df_no_data = df_signals[df_signals['Available_States'] == "No Data"]
display(df_no_data[['Event', 'Method', 'DataType']].head(10))

=== PARSING SUMMARY ===
Total Signals       : 7865
Signals with Enums  : 4620
Physical Signals    : 1371
Missing / No Data   : 1874

=== DATA TYPE DISTRIBUTION ===
DataType
uint8      6291
float32     513
uint16      275
uint32      250
N/A         226
boolean     136
uint64       55
sint32       45
sint16       32
sint64       26
sint8        10
UTF_8         6
Name: count, dtype: int64


=== UNIT DISTRIBUTION ===
Unit
CELSIUS       47
X_3           44
PERCENTAGE    43
V1            30
m1            28
              ..
l___mn         1
L___min        1
km___h         1
m2___s         1
X1___m         1
Name: count, Length: 100, dtype: int64


=== NO DATA SAMPLE ===


,Event,Method,DataType
2,SomeIpGadeSignal,gadeEvent,uint8
12,SomeIpMinVoltageReq,setMinVoltageAC,uint8
14,SomeIpMinVoltageReq,setMinVoltageWashing,uint8
16,SomeIpMinVoltageReq,setMinVoltageWiping,uint8
33,SomeIpPrimBattVoltRegulStatus,dcdcFaultypeEvent,uint8
34,SomeIpPrimBattVoltRegulStatus,dcdcStateEvent,uint8
36,SomeIpPrimBattVoltRegulStatus,dcdcTemperatureInfoEvent,uint8
38,SomeIpPrimBattVoltRegulStatus,voltageRegulationInfoEvent,uint8
39,SomeIpPrimBattVoltRegulStatus,voltageRegulationStateEvent,uint8
42,SomeIpPwtProducerLoad,lifeOnBoardPowerLimitationEvent,uint8


In [19]:
# 1. Force Pandas to show ALL columns and expand column width for readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60) 

# 2. Stitch the first 15 and last 15 rows together into a single view
display(pd.concat([df_signals.head(15), df_signals.tail(15)]))

# ---------------------------------------------------------
# ALTERNATIVE: If you prefer to view them as two completely separate tables
# ---------------------------------------------------------
# print("--- First 15 Signals ---")
# display(df_signals.head(15))

# print("--- Last 15 Signals ---")
# display(df_signals.tail(15))

,Cluster,SIF,Event,Method,Signal_String,DataType,Available_States,Min,Mid,Max,Factor,Offset,Unit
0,EthernetCluster,16414,SomeIpGadeSignal,GadeStatus,"""EthernetCluster::sif_16414::SomeIpGadeSignal::GadeStatus""",uint8,0: GADESTATUS_PARK_MODE | 1: GADESTATUS_LIFE_ON_BOARD | ...,0,3,7,N/A,N/A,N/A
1,EthernetCluster,16414,SomeIpGadeSignal,GadeStatusValueState,"""EthernetCluster::sif_16414::SomeIpGadeSignal::GadeStatu...",uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2,N/A,N/A,N/A
2,EthernetCluster,16414,SomeIpGadeSignal,gadeEvent,"""EthernetCluster::sif_16414::SomeIpGadeSignal::gadeEvent""",uint8,No Data,N/A,N/A,N/A,N/A,N/A,N/A
3,EthernetCluster,16414,SomeIpGadeSignal,gadeEventValueState,"""EthernetCluster::sif_16414::SomeIpGadeSignal::gadeEvent...",uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2,N/A,N/A,N/A
4,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueType,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::Voltag...",float32,Physical Value,10,13,16,1,0,Volt
5,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueTypeValueState,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::Voltag...",uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2,N/A,N/A,N/A
6,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::minVol...",float32,Physical Value,10,13,16,1,0,Volt
7,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::minVol...",float32,Physical Value,10,13,16,1,0,Volt
8,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1ValueState,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::minVol...",uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2,N/A,N/A,N/A
9,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue2,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::minVol...",float32,Physical Value,10,13,16,1,0,Volt


In [6]:
# 1. Remove exact duplicate rows across all columns
# df = df_signals.drop_duplicates()

# (Optional: If you want to ensure uniqueness specifically based on the Signal String itself)
df = df_signals.drop_duplicates(subset=['Signal_String'])

# 2. Count how many signals have Enums vs. how many don't
# Note: Our script explicitly labeled missing enums as the string "No Enums"
missing_enums_count = (df['Available_States'] == "No Enums").sum()
has_enums_count = (df['Available_States'] != "No Enums").sum()

print(f"Total Unique Signals: {len(df)}")
print(f"Signals WITH Enums: {has_enums_count}")
print(f"Signals WITHOUT Enums (False Positives/Missing): {missing_enums_count}")

# 3. (Bonus) View a quick breakdown of exactly which enums are present
# This shows the top 10 most frequent enum states across your database
print("\n--- Top 10 Most Common Enum Mappings ---")
print(df['Available_States'].value_counts().head(10))

Total Unique Signals: 7795
Signals WITH Enums: 4558
Signals WITHOUT Enums (False Positives/Missing): 3237

--- Top 10 Most Common Enum Mappings ---
Available_States
0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: VALUE_STATE_INVALID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  3374
No Enums                                                                                                                                          

In [7]:
# Display a dataframe containing ONLY the signals missing their Enums
df_missing = df[df['Available_States'] == "No Enums"]
df_missing

,Cluster,SIF,Event,Method,Signal_String,Available_States
2,EthernetCluster,16414,SomeIpGadeSignal,gadeEvent,"""EthernetCluster::sif_16414::SomeIpGadeSignal:...",No Enums
4,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueType,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
6,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
7,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
9,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue2,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
...,...,...,...,...,...,...
7855,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,Uuid6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7857,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,lsb6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7859,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,msb6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7861,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,setRHLBodyCmdRsp,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
